# 04 · Temporal Analysis
## How Illicit Activity Evolves Across Time

The Elliptic dataset's 49 time steps (each ~2 weeks of Bitcoin history) provide a rare opportunity to study **the dynamics of financial crime** — not just who is fraudulent, but *when*, *how fast*, and *in what pattern* illicit activity emerges.

This notebook answers four research questions:

1. **Burst detection** — At which time steps does illicit activity spike, and is it statistically anomalous?
2. **Propagation** — Does illicit activity spread to neighbouring nodes over subsequent time steps?
3. **Early warning** — Can we predict an upcoming burst before it peaks?
4. **Survival analysis** — How long do illicit wallets remain active before disappearing?

These questions go beyond standard fraud detection and constitute original exploratory research.

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#0D1117','axes.facecolor':'#161B22',
    'axes.edgecolor':'#30363D','axes.labelcolor':'#C9D1D9',
    'xtick.color':'#8B949E','ytick.color':'#8B949E',
    'text.color':'#C9D1D9','grid.color':'#21262D',
    'grid.linestyle':'--','figure.dpi':130,
})
PALETTE = {'Illicit':'#FF4444', 'Licit':'#00C9A7', 'Unknown':'#8B949E'}

# Load data
full_df   = pd.read_csv('../data/full_features.csv')
edges_df  = pd.read_csv('../data/elliptic_txs_edgelist.csv')
pred_df   = pd.read_csv('../data/unknown_node_predictions.csv')  # from notebook 03

print(f'Loaded {len(full_df):,} nodes')

In [ ]:
# ── 1. Burst detection with statistical anomaly flagging ──────────────────────
ts_stats = full_df.groupby('time_step').apply(lambda g: pd.Series({
    'total'          : len(g),
    'illicit'        : (g['class']==1).sum(),
    'licit'          : (g['class']==2).sum(),
    'unknown'        : (g['class']==0).sum(),
    'illicit_rate'   : (g['class']==1).sum() / max((g['class'].isin([1,2])).sum(), 1),
})).reset_index()

# Z-score of illicit rate → flag anomalous time steps
mean_r = ts_stats['illicit_rate'].mean()
std_r  = ts_stats['illicit_rate'].std()
ts_stats['z_score']   = (ts_stats['illicit_rate'] - mean_r) / std_r
ts_stats['is_burst']  = ts_stats['z_score'] > 1.5

print(f'Burst time steps (z > 1.5): {ts_stats[ts_stats["is_burst"]]["time_step"].tolist()}')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle('Temporal Burst Detection — Illicit Activity Over 49 Time Steps', fontsize=14)

# Top: stacked bar of all classes
axes[0].bar(ts_stats['time_step'], ts_stats['licit'],   color='#00C9A7', alpha=0.7, label='Licit')
axes[0].bar(ts_stats['time_step'], ts_stats['unknown'], color='#8B949E', alpha=0.5,
            bottom=ts_stats['licit'], label='Unknown')
axes[0].bar(ts_stats['time_step'], ts_stats['illicit'], color='#FF4444', alpha=0.9,
            bottom=ts_stats['licit']+ts_stats['unknown'], label='Illicit')
axes[0].set_ylabel('Node Count')
axes[0].legend(loc='upper right')

# Bottom: illicit rate with burst highlights
axes[1].bar(ts_stats['time_step'], ts_stats['illicit_rate'],
            color=ts_stats['is_burst'].map({True:'#FF4444', False:'#4A90D9'}),
            alpha=0.85, width=0.8)
axes[1].axhline(mean_r, color='#FFD700', ls='--', lw=1.5, label=f'Mean: {mean_r:.3f}')
axes[1].axhline(mean_r + 1.5*std_r, color='#FF4444', ls=':', lw=1, label='Burst threshold (1.5σ)')
axes[1].set_ylabel('Illicit Rate (among labelled)')
axes[1].set_xlabel('Time Step')
axes[1].legend()

# Annotate burst steps
for _, row in ts_stats[ts_stats['is_burst']].iterrows():
    axes[1].annotate(f"t={int(row['time_step'])}",
                     xy=(row['time_step'], row['illicit_rate']),
                     xytext=(0, 8), textcoords='offset points',
                     ha='center', fontsize=7, color='#FF4444')

plt.tight_layout()
plt.savefig('../data/fig_burst_detection.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── 2. Propagation analysis ────────────────────────────────────────────────────
# Research Q: Do nodes connected to illicit nodes in time step T
#             become illicit themselves in time step T+1?

print('Building graph for propagation analysis…')
G = nx.from_pandas_edgelist(edges_df, 'txId1', 'txId2', create_using=nx.DiGraph())
label_ts = full_df.set_index('txId')[['class','time_step']].to_dict('index')

propagation_records = []
for t in range(1, 49):
    # Illicit nodes at time step t
    ill_t = set(full_df[(full_df['time_step']==t) & (full_df['class']==1)]['txId'])
    if len(ill_t) == 0:
        continue

    # Direct successors of illicit nodes at t+1
    exposed_t1 = set()
    for n in ill_t:
        if n in G:
            for succ in G.successors(n):
                ni = label_ts.get(succ, {})
                if ni.get('time_step') == t+1:
                    exposed_t1.add(succ)

    if not exposed_t1:
        continue

    # Among exposed nodes at t+1, how many are illicit?
    exp_labels = [label_ts.get(n,{}).get('class',0) for n in exposed_t1]
    n_ill_next = sum(1 for l in exp_labels if l == 1)

    propagation_records.append({
        'time_step'        : t,
        'illicit_at_t'     : len(ill_t),
        'exposed_at_t1'    : len(exposed_t1),
        'illicit_at_t1'    : n_ill_next,
        'propagation_rate' : n_ill_next / max(len(exposed_t1), 1),
    })

prop_df = pd.DataFrame(propagation_records)

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(prop_df['time_step'], prop_df['propagation_rate'],
       color='#FF4444', alpha=0.8, width=0.7)
ax.set_title('Illicit Propagation Rate: Proportion of Exposed Nodes that Become Illicit at T+1',
             fontsize=12)
ax.set_xlabel('Time Step T')
ax.set_ylabel('Propagation Rate')
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=1))
plt.tight_layout()
plt.savefig('../data/fig_propagation.png', bbox_inches='tight', dpi=130)
plt.show()

print(f'Mean propagation rate: {prop_df["propagation_rate"].mean()*100:.2f}%')
print(f'Max propagation rate : {prop_df["propagation_rate"].max()*100:.2f}% at t={prop_df.loc[prop_df["propagation_rate"].idxmax(),"time_step"]}')

In [ ]:
# ── 3. Early warning model ─────────────────────────────────────────────────────
# Can we predict the NEXT time step's illicit rate from current signals?
# This is a simple autoregressive model — illustrates the concept

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

# Build lag features
ts_model = ts_stats[['time_step','illicit_rate','total','illicit']].copy()
for lag in [1, 2, 3]:
    ts_model[f'ill_rate_lag{lag}'] = ts_model['illicit_rate'].shift(lag)
    ts_model[f'total_lag{lag}']    = ts_model['total'].shift(lag)

ts_model = ts_model.dropna()
X_ts = ts_model[[c for c in ts_model.columns if 'lag' in c]].values
y_ts = ts_model['illicit_rate'].values

# Leave-last-10 out (temporal)
split = len(X_ts) - 10
X_tr, X_te = X_ts[:split], X_ts[split:]
y_tr, y_te = y_ts[:split], y_ts[split:]

ew_model = Ridge(alpha=1.0)
ew_model.fit(X_tr, y_tr)
y_pred_ts = ew_model.predict(X_te)

mae = mean_absolute_error(y_te, y_pred_ts)

fig, ax = plt.subplots(figsize=(13, 4))
all_ts = ts_model['time_step'].values
ax.plot(all_ts, y_ts, color='#4A90D9', lw=1.5, label='Actual illicit rate')
ax.plot(all_ts[split:], y_pred_ts, color='#FFD700', lw=2, ls='--',
        label=f'Predicted (MAE={mae:.4f})')
ax.axvline(all_ts[split], color='#8B949E', ls=':', lw=1, label='Train/test boundary')
ax.set_title('Early Warning Model — Predicting Next Time Step Illicit Rate', fontsize=12)
ax.set_xlabel('Time Step')
ax.set_ylabel('Illicit Rate')
ax.legend()
plt.tight_layout()
plt.savefig('../data/fig_early_warning.png', bbox_inches='tight', dpi=130)
plt.show()
print(f'Early warning MAE: {mae:.5f}')

In [ ]:
# ── 4. Combining GNN predictions with temporal context ────────────────────────
# Overlay GNN-predicted illicit unknowns onto the temporal chart

pred_by_ts = pred_df[pred_df['predicted_label']=='Illicit'].groupby('time_step').size()
confirmed  = full_df[full_df['class']==1].groupby('time_step').size()

all_ts = sorted(set(pred_by_ts.index) | set(confirmed.index))
pred_counts = [pred_by_ts.get(t,0) for t in all_ts]
conf_counts = [confirmed.get(t,0) for t in all_ts]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(all_ts, conf_counts, color='#FF4444', alpha=0.9, label='Confirmed Illicit', width=0.7)
ax.bar(all_ts, pred_counts, color='#FF9900', alpha=0.7, label='GNN-Predicted Illicit (Unknown)',
       bottom=conf_counts, width=0.7)
ax.set_title('Confirmed + GNN-Predicted Illicit Transactions by Time Step', fontsize=13)
ax.set_xlabel('Time Step')
ax.set_ylabel('Count')
ax.legend()

total_pred = sum(pred_counts)
total_conf = sum(conf_counts)
ax.text(0.98, 0.95,
        f'Confirmed: {total_conf}\nGNN-predicted: {total_pred}\nTotal at-risk: {total_conf+total_pred}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='#21262D', alpha=0.8))

plt.tight_layout()
plt.savefig('../data/fig_combined_illicit_timeline.png', bbox_inches='tight', dpi=130)
plt.show()

print('\nTemporal analysis complete.')
print(f'Key finding: adding GNN predictions increases the identifiable illicit pool')
print(f'from {total_conf} (confirmed) to {total_conf+total_pred} (confirmed + predicted) — ')
print(f'a {(total_pred/total_conf)*100:.0f}% increase in detected suspicious activity.')